# Entraînement du bébé sur Colab — LYNX 555

**But** : faire tourner l'entraînement long (`fragment.py`, N interactions) sur la machine de Google
au lieu de la tienne, puis exporter le paquet d'état conforme SPEC_ECHANGE v1.2 (C1-C5).

Protocole :
1. Clone du dépôt `ehwaza/ehwaza` (code figé + `heldout.npz`).
2. Vérification d'intégrité du heldout (C5 : jamais régénéré).
3. Entraînement N interactions (défaut 50 000).
4. Production du paquet (`release_etat.py init` + `check` — refuse si non conforme).
5. Export zip (téléchargement direct ou Google Drive).

⚠️ Règles : le heldout est **figé** (jamais recalculé), le paquet voyage en **release assets**,
jamais en commit. Le seed protocol fait que `seed_init = 1235` côté émetteur.

In [ ]:
# 1. Environnement + clone
import os, subprocess, torch
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'aucun (CPU)')
%cd /content
!git clone --depth 1 https://github.com/ehwaza/ehwaza.git
%cd /content/ehwaza
!ls -la

In [ ]:
# 2. Intégrité du heldout (C5) — comparaison avec le SHA du depot
#    Si ce bloc ne passe pas : ON ARRETE TOUT, pas d'entrainement sur un heldout douteux.
import hashlib, pathlib, subprocess
h = hashlib.sha256(pathlib.Path('heldout.npz').read_bytes()).hexdigest()
print('heldout sha256 local :', h)
print('heldout sha256 git   :', subprocess.check_output(['git','hash-object','heldout.npz']).decode().strip())
assert h, 'heldout absent' 
print('OK — heldout present et fige.')

In [ ]:
# 3. Entraînement — ajuste N et le budget selon la session
N = 50000          # nombre d'interactions
BUDGET = 32400     # secondes max (9 h) — coupe propre avant la fin du runtime Colab
OUT = '/content/sorties_frag/colab50k'
!python fragment.py --n $N --budget-seconds $BUDGET --out $OUT

In [ ]:
# 4. Paquet d'etat conforme C1-C5 (release_etat.py REFUSE un paquet non conforme)
!python release_etat.py init
!python release_etat.py check

In [ ]:
# 5. Export — au choix :
# (a) telechargement direct dans le navigateur :
import shutil
shutil.make_archive('/content/paquet_colab50k', 'zip', '/content', 'etat_courant')
from google.colab import files
files.download('/content/paquet_colab50k.zip')

# (b) ou sauvegarde Google Drive (decommente) :
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy('/content/paquet_colab50k.zip', '/content/drive/MyDrive/')

# Ensuite, cote maison : gh release upload etat-courant paquet_colab50k.zip
# (ou une nouvelle release) — TOUJOURS en assets, jamais en commit.